# Classification du jeu Imagenet-1K avec Alexnet

- Creation : *18/02/2025*

Constat de la performance de classification du modèle.
Utilisation de la définition du modèle et des poids disponibles dans PyTorch.

- [ ] Essayer AlexNet_Weights.IMAGENET1K_V1.transforms, ensemble des transformations prèdéfinie dédié à AlexNet entrainé sur Imagenet-1K.
- [ ] Evaluations de l'erreur de validation sur les jeux de données Imagenet-1K avec 1K images (imagenet-sample-images-master) et 50K images(imagenet_val_images) disponibles localement. 


# Module

In [ ]:
import os
from collections import Counter, defaultdict

import torch
import torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from datasets import DATASET_1, DATASET_2, CustomImageDataset, get_label_data_from_filename
from utils.alexnet_for_deconv import alexnet_for_deconv
from utils.utils_images import display_image_tensor as display_image_tensor_
from imagenet_labels import imagenet1K_labels_to_names

In [ ]:
# Spécifiquement pour un carnet de type Jupyter
def display_image_tensor(img_tensor, verbose=True):
    if display:
        display_image_tensor_(img_tensor, verbose=verbose, fn_display=display)

In [ ]:
test_debug = False
do_validation = False

# Device

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} device available")

# Chargement des données

Création du dataset sur les images

In [ ]:
DATASET = DATASET_1

In [ ]:
"""
from datasets import imagenet_mean, imagenet_std

geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])

transforms = T.Compose([
    geo_transforms,
    T.Lambda(lambda t: t/255.), # because read_image -> [0..255]
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])
""";

transforms = torchvision.models.AlexNet_Weights.IMAGENET1K_V1.transforms()

# Display the transforms
# Mais attention, contrairement à ce qui serait affiché, le resize est exécuté avant le crop
print("Transforms", transforms)

get_label_data = lambda f: get_label_data_from_filename(f, DATASET["path"])
dataset_path = DATASET["mounted_path"] if os.path.exists(DATASET["mounted_path"]) else DATASET["path"]
print("Dataset (name, path)", DATASET["name"], dataset_path)

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=True,
    only_label_idx=False,
    get_label_data=get_label_data,
    )

In [ ]:
dataset[1]

In [ ]:
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

Test dataloader

In [ ]:
if test_debug:
    i_element = 1
    batch = next(iter(dataloader))
    print("Dimension batch:", len(batch), end="\n\n")
    
    print("Première partie du Batch :", batch[0].size())
    print("Deuxième partie du Batch :", batch[1].size())
    print("Troisième partie du Batch :", len(batch[2]))
    print("Quatrième partie du Batch :", batch[3].size(), end="\n\n")
    
    print("Image :", batch[0][i_element].size())
    print("Idx etiquette :", batch[1][i_element].item())
    print("Code etiquette :", batch[2][i_element])
    print("Id file :", batch[3][i_element].item())

### Validation de la concordance des informations de classifications

In [ ]:
if do_validation:
    from ILSVRC2012_synsets import ILSVRC2012_synsets_idx_to_codes

    def read_imagenet_classes_ground_truth(file_path = "ILSVRC2012_validation_ground_truth.txt"):
        with open(file_path, 'r') as f:
            class_ids = [int(line.strip()) for line in f if line.strip().isdigit()]
        return class_ids

    # Exemple d'utilisation
    imagenet_classes_ground_truth = read_imagenet_classes_ground_truth()
    print("Nombre de lignes :", len(imagenet_classes_ground_truth))

    for filename, ground_truth_idx in zip(dataset.files, imagenet_classes_ground_truth):
        label_code, label_idx, _ = get_label_data_from_filename(filename, DATASET["path"])
        ground_truth_code = ILSVRC2012_synsets_idx_to_codes[ground_truth_idx]
        assert ground_truth_code == label_code, f"Discordance: {(ground_truth_code, ground_truth_idx)} != {(label_code, label_idx)} pour {filename}"
    print("Tous les labels concordent.")

# Chargement du modèle

In [ ]:
model_alexnet_deconv = alexnet_for_deconv(weights='IMAGENET1K_V1')
model_alexnet_deconv.eval()
model_alexnet_deconv.to(device)

In [ ]:
if test_debug:
    i_image = 10
    batch_input = batch[0][i_image].unsqueeze(dim=0)
    print("Batch input :", batch_input.size())
    output = model_alexnet_deconv.forward(batch_input.to(device))

    print(output.size())
    probabilities = torch.nn.functional.softmax(output, dim=1)
    predicted = probabilities.argmax(dim=1).to("cpu")
    print("predicted idx :", predicted)
    expected = batch[1][i_image]

    print("expected :", expected.tolist())

    #print(probabilities.argmax(dim=1).to("cpu") == batch[1][0])
    #print((probabilities.argmax(dim=1).to("cpu") == batch[1][0]).sum())

# Classification

Il faut espérer que l'indexation de la sortie du classifier correspond à celle des classes de l'ImageNet-1K récupérée dans la liste 1K.

In [ ]:
all_predictions_batch = []
all_expecteds_batch = []
for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    output = model_alexnet_deconv.forward(batch[0].to(device)) # To carefully transfer to CPU before append in list
    output = output.to("cpu")
    probabilities = torch.nn.functional.softmax(output, dim=0)
    top5_prob, top5_catid = torch.topk(probabilities, 5)
    all_predictions_batch.append((top5_prob.tolist(), top5_catid.tolist()))
    all_expecteds_batch.append(batch[1].tolist())

In [ ]:
from collections import Counter
distribution_computed_top1 = Counter()
distribution_computed_top5 = Counter()
distribution_expected = Counter()

right_top1 = 0
right_top5 = 0

for (predictions_batch_prob, predictions_batch_idx), expected_batch in zip(all_predictions_batch, all_expecteds_batch):
    predictions_batch_idx = torch.tensor(predictions_batch_idx)
    expected_batch = torch.tensor(expected_batch).reshape(-1, 1)
    expected_in_top1 = (predictions_batch_idx[:, 1] - expected_batch) == 0
    expected_in_top5 = (predictions_batch_idx - expected_batch) == 0

    right_top1 += expected_in_top1.sum().item()
    right_top5 += expected_in_top5.sum().item()

    distribution_computed_top1.update(predictions_batch_idx[:, 0].ravel().tolist())
    distribution_computed_top5.update(predictions_batch_idx.ravel().tolist())
    distribution_expected.update(expected_batch.ravel().tolist())


print(f"Justesse top1: {right_top1} ({right_top1 / len(dataset) * 100:.2f}%)")
print(f"Justesse top5: {right_top5} ({right_top5 / len(dataset) * 100:.2f}%)")

In [ ]:
print(distribution_computed_top1.total())
print(distribution_computed_top1.most_common(10))
print(distribution_computed_top5.total())
print(distribution_computed_top5.most_common(10))
list(distribution_expected.items())[:10]

Calcul des ratios par classe

Voir pour la matrice de confusion ?

In [ ]:
def counter_to_tensor(counter, size=10):
    a = [0] * size
    for k, v in counter.items():
        a[k] = v
    return torch.tensor(a)

distribution_expected_t = counter_to_tensor(distribution_expected, len(imagenet1K_labels_to_names))
distribution_computed_t = counter_to_tensor(distribution_computed_top1, len(imagenet1K_labels_to_names))

print((distribution_computed_t - distribution_expected_t).abs().sum())

print("Equal distrib count", (distribution_computed_t == distribution_expected_t).sum().item())
print("Equal distrib ratio", (distribution_computed_t == distribution_expected_t).sum() / len(dataset))


#print("Equal distrib ratio by class", (distribution_computed_t - distribution_expected_t).abs() / torch.where(distribution_expected_t == 0, torch.tensor(1), distribution_expected_t))

#print(distribution_computed_t)
#print(distribution_expected_t)